# Lesson 01 - AI 에이전트 소개

**초보자를 위한 AI 에이전트** 코스의 첫 번째 레슨에 오신 것을 환영합니다!

**AI 에이전트**는 대형 언어 모델(LLM)을 추론 엔진으로 사용하고, *행동*을 취할 수 있는 프로그램으로서 — API를 호출하거나, 데이터베이스를 조회하거나, 코드를 실행하는 등 — 사용자를 대신해 목표를 달성합니다.

이 노트북에서는 첫 번째 에이전트를 구축할 것입니다: 휴가 여행지를 추천하는 **여행 에이전트**입니다. 이 과정에서 다음을 배우게 됩니다:

1. **Microsoft Agent Framework**를 사용하여 Azure AI Foundry Agent Service에 연결하는 방법.
2. 에이전트가 호출할 수 있는 평범한 Python 함수인 **도구**를 에이전트에 제공하는 방법.
3. 에이전트를 실행하고 그 응답을 검사하는 방법.
4. 에이전트의 응답을 토큰 단위로 스트리밍하는 방법.


## 설정

이 노트북을 실행하기 전에 다음을 확인하세요:

1. **배포된 채팅 모델이 포함된 Azure AI Foundry 프로젝트** (예: `gpt-4o-mini`).
2. **Azure CLI에 로그인** — 터미널에서 `az login`을 실행하세요.
3. **필요한 환경 변수 설정:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry 프로젝트 엔드포인트.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 배포된 모델 이름.

아래 셀은 필요한 Python 패키지를 설치합니다.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 첫 번째 에이전트 만들기

에이전트에는 두 가지가 필요합니다:

- 에이전트가 *누구인지*와 *어떻게 행동해야 하는지* 알려주는 **명령어**(시스템 프롬프트).
- 에이전트가 정보를 검색하거나 작업을 수행하기 위해 호출할 수 있는 `@tool`로 장식된 Python 함수인 **도구**.

아래에서는 인기 있는 휴양지 목록을 반환하는 간단한 도구를 정의합니다. 사용자가 여행 추천을 요청할 때 에이전트가 이 도구를 사용할 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 스트리밍 응답

보다 상호작용적인 경험을 위해 에이전트의 응답을 **스트리밍**할 수 있습니다. 전체 답변을 기다리는 대신, 에이전트가 생성되는 텍스트 조각을 순차적으로 전달합니다. 이는 출력물을 실시간으로 표시하려는 채팅 인터페이스에서 특히 유용합니다.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 요약

이 강의에서는 다음을 배웠습니다:

- `AzureAIProjectAgentProvider`를 통해 Azure AI Foundry Agent Service에 연결하는 **프로바이더 생성** 방법.
- 에이전트가 Python 함수를 호출할 수 있도록 `@tool` 데코레이터를 사용하여 **도구 정의** 방법.
- 사용자 메시지로 **에이전트를 실행하고 응답을 출력**하는 방법.
- 실시간 출력을 위한 **응답 스트리밍** 방법.

다음 강의에서는 에이전트 프레임워크를 더 깊이 탐구하고, 에이전트에 보다 강력한 도구와 다단계 추론 기능을 제공하는 방법을 배울 것입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확성이 포함될 수 있음을 유의해 주시기 바랍니다. 원문이 작성된 언어의 원본 문서를 권위 있는 출처로 간주해야 합니다. 중요한 정보의 경우 전문 인력에 의한 번역을 권장합니다. 본 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
